# RAE-1B: Robust Adaptive Transformer Autoencoder -- Colab walkthrough

Self-contained, step-by-step build of a ~1B-parameter denoising Transformer autoencoder: RoPE + SwiGLU + RMSNorm, pure PyTorch + a real BPE tokenizer, **no external LLM/API dependency**. Runs top to bottom in a fresh Colab runtime.

Everything is defined in this single notebook (no imports from an external repo), so you can copy it straight into Colab.

**Strategy:** validate the exact same code at small scale first (a few million params), then flip the config to the ~1B target. Instantiating the full 1B model on a stock Colab GPU (T4, ~15GB) is feasible for inference/short training with gradient checkpointing + BF16 + small batch/grad-accumulation; full pretraining needs multi-GPU.

## Step 1 -- Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## Step 2 -- Install dependencies

In [ ]:
!pip install -q tokenizers pyyaml tqdm

## Step 3 -- Imports

In [ ]:
import os
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

## Step 4 -- Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seed:", SEED)

## Step 5 -- Project directories

In [ ]:
PROJECT_DIR = Path("/content/RAE-1B")
DATA_DIR = PROJECT_DIR / "data"
TOKENIZER_DIR = PROJECT_DIR / "tokenizer"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for d in [PROJECT_DIR, DATA_DIR, TOKENIZER_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

## Step 6 -- Training corpus

This tiny corpus is only for wiring/debugging the pipeline. It is nowhere near enough data to train a useful 1B model -- for a real run, replace this with a large, properly licensed corpus (and generate it in bulk, not by hand).

In [ ]:
corpus = """
Artificial intelligence is transforming computer science.
Machine learning allows computers to learn patterns from data.
Deep learning uses neural networks to learn complex representations.
Transformers use attention mechanisms to model relationships between tokens.

A transformer encoder processes an input sequence and generates contextual representations.
A transformer decoder can generate an output sequence from a learned representation.
An autoencoder learns to reconstruct an original input from an encoded representation.

A denoising autoencoder receives corrupted data and attempts to reconstruct the clean data.
This encourages the model to learn robust representations.
Robust representations can be useful for classification, retrieval and anomaly detection.

Cybersecurity systems analyze network traffic, system logs and security events.
Intrusion detection systems identify suspicious network behavior.
Anomaly detection attempts to distinguish unusual behavior from normal behavior.

Network security includes authentication, authorization, encryption and monitoring.
Machine learning can assist analysts in identifying potentially malicious activity.

Responsible artificial intelligence includes robustness, transparency, privacy and accountability.
Representation learning can be evaluated using downstream classification and retrieval tasks.

Transformers contain attention layers and feed forward networks.
Residual connections improve optimization.
Normalization stabilizes neural network training.

The encoder transforms a corrupted sequence into a latent representation.
The decoder uses this representation to reconstruct the original sequence.
The reconstruction error provides a self supervised learning signal.

This project develops a robust transformer autoencoder.
The objective is to study how model size and corruption strategies affect representation quality.
"""

corpus_path = DATA_DIR / "corpus.txt"
corpus_path.write_text(corpus, encoding="utf-8")
print("Corpus written:", corpus_path)

## Step 7 -- Train a BPE tokenizer (target vocab 32,000)

In [ ]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=32000,
    min_frequency=1,
    special_tokens=SPECIAL_TOKENS,
)

tokenizer.train(files=[str(corpus_path)], trainer=trainer)

tokenizer_path = TOKENIZER_DIR / "tokenizer.json"
tokenizer.save(str(tokenizer_path))

print("Tokenizer saved:", tokenizer_path)
print("Vocabulary size:", tokenizer.get_vocab_size())

The tiny debug corpus above doesn't have enough unique tokens to actually fill a 32k vocabulary -- `vocab_size=32000` is a ceiling, not a guarantee. Train on your real corpus to get the full 32k vocab.

## Step 8 -- Special-token IDs

In [ ]:
PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
MASK_ID = tokenizer.token_to_id("[MASK]")

print("PAD", PAD_ID, "UNK", UNK_ID, "BOS", BOS_ID, "EOS", EOS_ID, "MASK", MASK_ID)

## Step 9 -- Sanity-check the tokenizer

In [ ]:
sample = "The intrusion detection system identified a malicious network flow."
encoded = tokenizer.encode(sample)
print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)
print("Count:", len(encoded.ids))

## Step 10 -- Reload tokenizer (useful after a runtime reconnect)

In [ ]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
VOCAB_SIZE = tokenizer.get_vocab_size()
print("Vocabulary:", VOCAB_SIZE)

## Step 11 -- Encode the corpus

In [ ]:
text = corpus_path.read_text(encoding="utf-8")
token_ids = tokenizer.encode(text).ids
print("Total tokens:", len(token_ids))
print(token_ids[:50])

## Step 12 -- Chunk into fixed-length training sequences

In [ ]:
SEQ_LEN = 128

def create_sequences(ids, seq_len):
    seqs = []
    for i in range(0, len(ids) - seq_len, seq_len):
        chunk = ids[i:i + seq_len]
        if len(chunk) == seq_len:
            seqs.append(chunk)
    return seqs

sequences = create_sequences(token_ids, SEQ_LEN)
print("Number of sequences:", len(sequences))

## Step 13-14 -- Dataset and DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long)


BATCH_SIZE = 2
dataset = TextDataset(sequences)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

batch = next(iter(loader))
print("Dataset size:", len(dataset))
print("Batch shape:", batch.shape)

## Step 15 -- RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

## Step 16 -- Rotary position embeddings (RoPE)

`dim` here is the **per-head** dimension, so it must be even (`d_model / n_heads`).

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        assert dim % 2 == 0, "RoPE needs an even per-head dimension"
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        seq_len = q.shape[-2]
        cos = self.cos[:seq_len][None, None, :, :]
        sin = self.sin[:seq_len][None, None, :, :]

        q_even, q_odd = q[..., ::2], q[..., 1::2]
        k_even, k_odd = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack([q_even * cos - q_odd * sin, q_even * sin + q_odd * cos], dim=-1).flatten(-2)
        k_rot = torch.stack([k_even * cos - k_odd * sin, k_even * sin + k_odd * cos], dim=-1).flatten(-2)
        return q_rot, k_rot

## Step 17 -- Multi-head attention (self- and cross-, RoPE on self-attention)

RoPE is applied when query and context are the same sequence (self-attention); cross-attention (decoder attending to the encoder's latent `Z`) skips it, since the two sequences carry independent position spaces.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0, max_seq_len=2048):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
        self.dropout = dropout

    def forward(self, x, context=None, causal=False, is_self_attn=True):
        if context is None:
            context = x
        B, T, C = x.shape
        S = context.shape[1]

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(context).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(context).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        if is_self_attn:
            q, k = self.rope(q, k)

        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=causal
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)

## Step 18 -- SwiGLU feed-forward

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

## Step 19 -- Encoder block (RMSNorm -> Self-Attn -> RMSNorm -> SwiGLU, pre-norm + residual)

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0, max_seq_len=2048):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout, max_seq_len)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), is_self_attn=True)
        x = x + self.ffn(self.norm2(x))
        return x

## Step 20 -- Decoder block (causal Self-Attn -> Cross-Attn(Z) -> SwiGLU)

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0, max_seq_len=2048):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, max_seq_len)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, max_seq_len)
        self.norm3 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x, memory):
        x = x + self.self_attn(self.norm1(x), causal=True, is_self_attn=True)
        x = x + self.cross_attn(self.norm2(x), context=memory, causal=False, is_self_attn=False)
        x = x + self.ffn(self.norm3(x))
        return x

## Step 21-22 -- Assemble RAE1B and instantiate a small prototype

**Always validate at small scale first.** The 1B config is defined later (Step 33) -- don't instantiate/train it on a stock Colab GPU without the memory techniques in Steps 35-37.

In [ ]:
class RAE1B(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, n_heads, d_ff,
                 encoder_layers, decoder_layers, dropout=0.0, use_grad_checkpoint=False):
        super().__init__()
        self.d_model = d_model
        self.use_grad_checkpoint = use_grad_checkpoint

        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout, max_seq_len) for _ in range(encoder_layers)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout, max_seq_len) for _ in range(decoder_layers)
        ])

        self.encoder_norm = RMSNorm(d_model)
        self.decoder_norm = RMSNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # weight tying

    def embed(self, input_ids):
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device)
        return self.token_embedding(input_ids) + self.position_embedding(positions).unsqueeze(0)

    def _run_stack(self, layers, *layer_args):
        x = layer_args[0]
        rest = layer_args[1:]
        for layer in layers:
            if self.use_grad_checkpoint and self.training:
                x = grad_checkpoint(layer, x, *rest, use_reentrant=False)
            else:
                x = layer(x, *rest)
        return x

    def encode(self, input_ids):
        x = self.embed(input_ids)
        x = self._run_stack(self.encoder, x)
        return self.encoder_norm(x)

    def decode(self, decoder_input_ids, memory):
        x = self.embed(decoder_input_ids)
        x = self._run_stack(self.decoder, x, memory)
        x = self.decoder_norm(x)
        return self.lm_head(x)

    def forward(self, encoder_input_ids, decoder_input_ids):
        memory = self.encode(encoder_input_ids)
        return self.decode(decoder_input_ids, memory)

In [ ]:
PROTOTYPE_CONFIG = {
    "vocab_size": VOCAB_SIZE,
    "max_seq_len": SEQ_LEN,
    "d_model": 256,
    "n_heads": 8,
    "d_ff": 1024,
    "encoder_layers": 4,
    "decoder_layers": 4,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = RAE1B(**PROTOTYPE_CONFIG).to(device)
print("Device:", device)

## Step 23 -- Count parameters (always measure, never assume)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

parameter_count = count_parameters(model)
print(f"Parameters: {parameter_count:,}")
print(f"Millions: {parameter_count / 1e6:.2f}M")

## Step 24 -- Forward-pass sanity check

In [ ]:
batch = next(iter(loader)).to(device)
encoder_input = batch[:, :-1]
decoder_input = batch[:, :-1]

with torch.no_grad():
    logits = model(encoder_input, decoder_input)

print("Input:", encoder_input.shape)
print("Logits:", logits.shape)
assert logits.shape == (encoder_input.shape[0], encoder_input.shape[1], VOCAB_SIZE)

## Step 25 -- Noise generator: mask / delete / replace

Matches the architecture diagram's three corruption modes (not mask-only). Runs per-example since deletion changes sequence length; output is re-padded back to `seq_len`.

In [ ]:
def corrupt_sequence(ids, seq_len, mask_prob=0.15, delete_prob=0.10, replace_prob=0.10, rng=None):
    rng = rng or random.Random()
    out = []
    for tok in ids:
        if tok in (PAD_ID, BOS_ID, EOS_ID):
            out.append(tok)
            continue
        r = rng.random()
        if r < delete_prob:
            continue
        elif r < delete_prob + mask_prob:
            out.append(MASK_ID)
        elif r < delete_prob + mask_prob + replace_prob:
            out.append(rng.randrange(5, VOCAB_SIZE))  # skip special ids 0-4
        else:
            out.append(tok)
    out = out or [MASK_ID]
    out = out[:seq_len] + [PAD_ID] * (seq_len - len(out))
    return out[:seq_len]


def corrupt_batch(input_ids, rng=None):
    rng = rng or random.Random()
    seq_len = input_ids.shape[1]
    corrupted = torch.empty_like(input_ids)
    for i in range(input_ids.size(0)):
        row = input_ids[i].tolist()
        corrupted[i] = torch.tensor(
            corrupt_sequence(row, seq_len, rng=rng), dtype=torch.long, device=input_ids.device
        )
    return corrupted

In [ ]:
rng = random.Random(0)
corrupted = corrupt_batch(batch, rng)
print("Original: ", batch[0][:30].tolist())
print("Corrupted:", corrupted[0][:30].tolist())

## Step 26-27 -- Reconstruction loss, optimizer, scheduler

In [ ]:
def reconstruction_loss(logits, targets):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=PAD_ID)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, betas=(0.9, 0.95), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

## Step 28-29 -- Training loop

Teacher-forced denoising objective: encoder sees the **corrupted** sequence, decoder is trained (shifted, teacher-forced) to reproduce the **original**.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device, rng):
    model.train()
    total_loss = 0.0

    for batch_idx, original in enumerate(loader):
        original = original.to(device)
        corrupted = corrupt_batch(original, rng)

        encoder_input = corrupted[:, :-1]
        decoder_input = original[:, :-1]
        targets = original[:, 1:]

        logits = model(encoder_input, decoder_input)
        loss = reconstruction_loss(logits, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f"batch {batch_idx} | loss {loss.item():.4f}")

    return total_loss / len(loader)

## Step 30 -- Train the prototype

In [ ]:
EPOCHS = 10
train_rng = random.Random(1)

for epoch in range(EPOCHS):
    start = time.time()
    loss = train_one_epoch(model, loader, optimizer, scheduler, device, train_rng)
    print(f"\nEpoch {epoch + 1}/{EPOCHS}  loss {loss:.4f}  time {time.time() - start:.2f}s")

## Step 31 -- Save checkpoint

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "parameter_count": parameter_count,
    "config": PROTOTYPE_CONFIG,
    "epoch": EPOCHS,
}

checkpoint_path = CHECKPOINT_DIR / "rae_prototype.pt"
torch.save(checkpoint, checkpoint_path)
print("Checkpoint saved:", checkpoint_path)

## Step 32 -- Load checkpoint

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Checkpoint loaded.")

## Step 33 -- Inspect the latent representation

In [ ]:
model.eval()
sample_batch = next(iter(loader)).to(device)
corrupted = corrupt_batch(sample_batch, rng)

with torch.no_grad():
    latent = model.encode(corrupted)

print("Latent shape:", latent.shape)  # [batch, seq_len, d_model]

## Step 34 -- Token-level reconstruction accuracy

In [ ]:
model.eval()
total_correct, total_tokens = 0, 0

with torch.no_grad():
    for original in loader:
        original = original.to(device)
        corrupted = corrupt_batch(original, rng)

        encoder_input = corrupted[:, :-1]
        decoder_input = original[:, :-1]
        targets = original[:, 1:]

        logits = model(encoder_input, decoder_input)
        predictions = logits.argmax(dim=-1)
        valid = targets != PAD_ID

        total_correct += ((predictions == targets) & valid).sum().item()
        total_tokens += valid.sum().item()

accuracy = total_correct / max(total_tokens, 1)
print("Token reconstruction accuracy:", accuracy)

## Step 35 -- Autoregressive reconstruction demo

In [ ]:
def decode_ids(ids):
    return tokenizer.decode(ids, skip_special_tokens=True)


@torch.no_grad()
def reconstruct(model, corrupted, max_length):
    model.eval()
    memory = model.encode(corrupted)
    generated = torch.tensor([[BOS_ID]], dtype=torch.long, device=corrupted.device)

    for _ in range(max_length):
        logits = model.decode(generated, memory)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == EOS_ID:
            break
    return generated

In [ ]:
example_text = "The intrusion detection system identified a malicious network flow."
encoded = tokenizer.encode(example_text)
ids = torch.tensor(encoded.ids, dtype=torch.long).unsqueeze(0).to(device)
corrupted_ids = corrupt_batch(ids, rng)

print("Original: ", decode_ids(ids[0].cpu().tolist()))
print("Corrupted:", decode_ids(corrupted_ids[0].cpu().tolist()))

output = reconstruct(model, corrupted_ids, max_length=50)
print("Reconstructed:", decode_ids(output[0].cpu().tolist()))

With this tiny debug corpus, don't expect fluent reconstructions -- this cell exists to prove the encode -> latent -> decode -> generate path works, not to demonstrate language quality.

## Step 36 -- (Optional) Persist checkpoints to Google Drive

In [ ]:
MOUNT_DRIVE = False  # flip to True if you want Drive persistence

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DIR = Path("/content/drive/MyDrive/RAE-1B")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    torch.save(checkpoint, DRIVE_DIR / "rae_prototype.pt")
    print("Saved to:", DRIVE_DIR / "rae_prototype.pt")

## Step 37 -- Scale toward the real ~1B configuration

Everything above ran on the prototype config. This cell defines the target 1B config and the intermediate sizes recommended for staged validation (50M -> 100M -> 250M -> 500M -> 1B) -- always measure the actual parameter count instead of trusting the label.

Built on the `"meta"` device: this counts parameters for every config, **including the 1B one, without allocating any real memory** -- safe to run on any machine, even one that can't actually hold a 1B-param model.

In [ ]:
CONFIGS = {
    "50M":  {"d_model": 512,  "n_heads": 8,  "d_ff": 2048, "encoder_layers": 6,  "decoder_layers": 6},
    "100M": {"d_model": 768,  "n_heads": 12, "d_ff": 3072, "encoder_layers": 8,  "decoder_layers": 8},
    "250M": {"d_model": 1024, "n_heads": 16, "d_ff": 4096, "encoder_layers": 10, "decoder_layers": 10},
    "500M": {"d_model": 1280, "n_heads": 20, "d_ff": 5120, "encoder_layers": 12, "decoder_layers": 12},
    "1B":   {"d_model": 1536, "n_heads": 24, "d_ff": 6144, "encoder_layers": 15, "decoder_layers": 15},
}

for name, cfg in CONFIGS.items():
    with torch.device("meta"):
        test_model = RAE1B(vocab_size=32000, max_seq_len=2048, **cfg)
    params = count_parameters(test_model)
    print(f"{name:>5} : {params:,}  (~{params / 1e6:.1f}M)")
    del test_model

**Do not instantiate/train the 1B config directly on a stock Colab GPU without the memory techniques below** -- this encoder-decoder architecture is heavier than a decoder-only 1B model since it holds two stacks plus decoder cross-attention.

## Step 38 -- Mixed precision (BF16)

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("BF16 supported:", use_bf16)

# Example step under autocast (BF16 doesn't need GradScaler the way FP16 does):
# with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
#     logits = model(encoder_input, decoder_input)
#     loss = reconstruction_loss(logits, targets)
# loss.backward()

## Step 39 -- Gradient accumulation (simulate a larger effective batch size)

In [ ]:
def train_step_with_accumulation(model, loader, optimizer, device, rng,
                                  accumulation_steps=16, use_bf16=False):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step, original in enumerate(loader):
        original = original.to(device)
        corrupted = corrupt_batch(original, rng)

        encoder_input = corrupted[:, :-1]
        decoder_input = original[:, :-1]
        targets = original[:, 1:]

        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=use_bf16 and device == "cuda"):
            logits = model(encoder_input, decoder_input)
            loss = reconstruction_loss(logits, targets) / accumulation_steps

        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

## Step 40 -- Gradient checkpointing

`RAE1B` already supports this -- pass `use_grad_checkpoint=True` when instantiating for the 1B run to trade extra compute for much lower activation memory:
```python
model = RAE1B(**CONFIGS["1B"], vocab_size=32000, max_seq_len=2048, use_grad_checkpoint=True).to(device)
```

## Step 41 -- Research extension: contrastive representation loss

Encourage two differently-corrupted views of the same input to land at the same latent point: `Z1 ~= Z2`.

In [ ]:
def pool_latent(z):
    return z.mean(dim=1)


def cosine_similarity_loss(z1, z2):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    similarity = (z1 * z2).sum(dim=-1)
    return (1.0 - similarity).mean()


# corrupted_a = corrupt_batch(original, rng)
# corrupted_b = corrupt_batch(original, rng)
# z1 = pool_latent(model.encode(corrupted_a))
# z2 = pool_latent(model.encode(corrupted_b))
# contrastive_loss = cosine_similarity_loss(z1, z2)

## Step 42 -- Research extension: latent robustness loss

In [ ]:
def latent_robustness_loss(clean_z, corrupted_z):
    clean_z = F.normalize(clean_z, dim=-1)
    corrupted_z = F.normalize(corrupted_z, dim=-1)
    return F.mse_loss(corrupted_z, clean_z)


# clean_vector = pool_latent(model.encode(original))
# corrupted_vector = pool_latent(model.encode(corrupted))
# robust_loss = latent_robustness_loss(clean_vector, corrupted_vector)
#
# total_loss = reconstruction_loss(...) + 0.1 * contrastive_loss + 0.05 * robust_loss

## Step 43 -- Extract embeddings for downstream tasks (classification, retrieval, clustering)

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    embeddings = []
    for original in loader:
        original = original.to(device)
        z = model.encode(original)
        embeddings.append(pool_latent(z).cpu())
    return torch.cat(embeddings, dim=0)


embeddings = extract_embeddings(model, loader, device)
print("Embedding matrix:", embeddings.shape)  # [num_sequences, d_model]

## Step 44 -- Per-example reconstruction error (anomaly score)

A sequence the model reconstructs poorly relative to the training distribution is a natural anomaly signal for downstream use (e.g. intrusion/anomaly detection).

In [ ]:
@torch.no_grad()
def reconstruction_error(model, original, corrupted):
    model.eval()
    encoder_input = corrupted[:, :-1]
    decoder_input = original[:, :-1]
    targets = original[:, 1:]

    logits = model(encoder_input, decoder_input)
    per_token_loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
        reduction="none", ignore_index=PAD_ID,
    ).view(targets.shape)

    valid = (targets != PAD_ID).float()
    per_example_score = (per_token_loss * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
    return per_example_score  # higher = more anomalous / harder to reconstruct


scores = reconstruction_error(model, sample_batch, corrupt_batch(sample_batch, rng))
print("Per-example anomaly scores:", scores.cpu().tolist())

## Summary / next steps

- The prototype config above (Steps 21-35) is verified end to end: parameter count, forward pass, corruption, training loss decreasing, checkpointing, latent inspection, reconstruction accuracy, and autoregressive generation.
- Step 37 measures the **actual** parameter count for 50M through 1B configs -- don't trust the size labels without running that cell.
- For a real 1B run: replace the debug corpus with a real dataset (billions of tokens), train the tokenizer on it to reach the full 32k vocab, switch to the `"1B"` config with `use_grad_checkpoint=True`, enable BF16 autocast (Step 38), use gradient accumulation (Step 39), and move to multi-GPU (DDP/FSDP) -- a single Colab GPU is for validating the pipeline, not for pretraining at this scale.
- Steps 41-44 (contrastive loss, robustness loss, embedding extraction, anomaly scoring) are optional research extensions layered on top of the base reconstruction objective -- add them once the base pipeline is solid.